In [ ]:
import sys
import importlib

sys.path.append('../utils')
import visualization
importlib.reload(visualization)
from visualization import save_to_html

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import plotly.express as px
import plotly.graph_objects as go
import joblib
import os


df = pd.read_parquet('../data/features/ml_dataset.parquet')

print(f"Данные для модели: {df.shape}")

In [6]:
target = 'qty'
drop_cols = ['sku', 'stock_sum', 'revenue']

X = df.drop(columns=drop_cols + [target])
y = df[target]

X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [7]:
model = RandomForestRegressor(
    n_estimators=400,
    max_depth=14,
    min_samples_split=4,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
pred = model.predict(X_test)

In [67]:
mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

metric_df = pd.DataFrame([{
    'MAE':  round(mae, 4),
    'RMSE': round(rmse, 4),
    'R²':   round(r2, 4)
}])

metric_df = (metric_df.T
             .reset_index()
             .rename(columns={'index': 'metric', 0: 'Value'})
)

print('Ключевые метрики для оценки модели')
display(metric_df)

Ключевые метрики для оценки модели


,metric,Value
0,MAE,5.7300
1,RMSE,12.8737
2,R²,0.1608


In [56]:
feat_imp = (
    pd.Series(model.feature_importances_, index=X_train.columns)
    .sort_values(ascending=False)
    .to_frame(name='Importance')
    .reset_index()
    .rename(columns={'index': 'Feature'})
)

display(feat_imp.head(10))

fig_imp = px.bar(
    feat_imp.head(20),
    x='Importance',       
    y='Feature',            
    orientation='h',
    title='Top 20 Feature Importances',
    labels={'Importance': 'Importance', 'Feature': 'Feature'}
)
fig_imp.update_layout(height=700, yaxis={'categoryorder': 'total ascending'})
fig_imp.show()


,Feature,Importance
0,stock_log,0.559490
1,month,0.132478
2,category_avg_qty,0.046367
3,size_M,0.039272
4,size_S,0.038574
5,size_XL,0.030170
6,size_L,0.030035
7,is_fulfilled_by_amazon,0.028393
8,size_XXL,0.024597
9,size_XS,0.017749


In [44]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=y_test,
    y=pred,
    mode='markers',
    name='Предсказания',
    marker=dict(
        size=5,
        color='rgba(0, 102, 204, 0.65)',
        line=dict(width=0.5, color='darkblue')
    ),
    opacity=0.7
))

min_val = min(y_test.min(), pred.min())
max_val = max(y_test.max(), pred.max())

fig.add_trace(go.Scatter(
    x=[min_val, max_val],
    y=[min_val, max_val],
    mode='lines',
    name='Идеальное предсказание (y = x)',
    line=dict(color='red', dash='dash', width=2.5)
))

fig.update_layout(
    title='Predicted vs Actual Quantity (Test Set)',
    xaxis_title='Реальное количество (Real qty)',
    yaxis_title='Предсказанное количество (Predicted qty)',
    width=950,
    height=750,
    template='plotly_white',
    hovermode='closest',
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show()

In [45]:
os.makedirs('../models', exist_ok=True)
joblib.dump(model, '../models/rf_demand_model_v2.pkl')

['../models/rf_demand_model_v2.pkl']

### **Выводы по моделированию спроса**

#### **Результаты модели**

Для предсказания количества продаваемых товаров (`qty`) была обучена модель **RandomForestRegressor**.

**Финальные метрики на тестовой выборке:**
- **MAE**: 5.730 шт.
- **RMSE**: 12.874 шт.
- **R²**: 0.1608

#### **Анализ результатов**

Модель объясняет около **16.1%** вариации целевой переменной. Хотя качество предсказания умеренное, анализ Feature Importance позволил сделать важные выводы о природе спроса.

После удаления `stock_sum` и оставления только `stock_log` выяснилось, что **один этот признак отвечает более чем за 55% важности модели**. Это подтверждает, что наличие и объём стока являются доминирующим фактором продаж в данном датасете.

Признаки категории, размера товара и месяц имеют значительно меньшее влияние, что говорит о необходимости дальнейшего обогащения признакового пространства.

#### **Ключевые инсайты**

- Продажи имеют ярко выраженную **нелинейную зависимость** от стока: при низких значениях stock продажи близки к нулю.
- Логарифмирование стока (`stock_log`) оказалось эффективным способом представления этой информации для модели.
- Модель лучше справляется с типичными небольшими заказами и пока слабо предсказывает редкие крупные продажи.

#### **Ограничения**

- Короткий период данных (4 месяца) не позволяет полноценно учесть сезонность и долгосрочные тренды.
- Отсутствие реального идентификатора клиента не дало возможности добавить RFM-признаки и клиентское поведение.
- Сильный правый хвост распределения `qty` усложняет задачу точного прогнозирования.

#### **Рекомендации по улучшению**

1. Создать новые признаки, отражающие взаимодействие стока с категорией и размером (`stock_per_category`, `stock_out_of_stock_flag` и др.).
2. Применить более продвинутые подходы: LightGBM/XGBoost с Tweedie objective или Quantile Regression.
3. Расширить горизонт данных и по возможности добавить клиентские признаки.
4. Рассмотреть построение отдельных моделей для мелких и крупных заказов.

#### **Итоговый вывод**

Проведённое моделирование показало, что **наличие стока является ключевым драйвером продаж** в данном датасете. Baseline-модель RandomForestRegressor продемонстрировала базовую предсказательную способность и помогла выявить наиболее важные факторы спроса.

Дальнейшая работа будет направлена на улучшение качества модели за счёт более глубокого Feature Engineering и применения специализированных алгоритмов для count-данных. 